In [1]:
import os
import pandas as pd
import dask.dataframe as dd
from pathlib import Path

# Set your input and output directories
input_directory = "/Volumes/T7/FCAS_RAISE1SEC-Volume-Bids/"
output_directory = "/Volumes/T7/Filtered_Parquet_Files/"

# Explore a single file first to understand its structure
sample_file = "/Volumes/T7/FCAS_RAISE1SEC-Volume-Bids/PUBLIC_DVD_BIDPEROFFER1_202402010000.CSV"

# Check if file exists
if os.path.exists(sample_file):
    print(f"Exploring sample file: {sample_file}")
    
    # Use pandas to read a few rows, skipping the first row
    pd_sample = pd.read_csv(sample_file, skiprows=1, nrows=5)
    print("\nSample using pandas (first 5 rows, skipping first row):")
    print(pd_sample.head())
    print("\nColumn names:", pd_sample.columns.tolist())
    
    # Now use dask to read the file, also skipping the first row
    ddf = dd.read_csv(sample_file, skiprows=1)
    
    # Compute and display the first few rows
    print("\nSample using dask (first 5 rows, skipping first row):")
    print(ddf.head())
    
    # Check for potential BIDTYPE column
    for col in pd_sample.columns:
        if 'BIDTYPE' in col.upper():
            print(f"\nFound potential BIDTYPE column: {col}")
            # Show unique values in this column
            unique_values = ddf[col].unique().compute()
            print(f"Unique values in {col}: {unique_values}")
else:
    print(f"Sample file not found: {sample_file}")
    print("Please provide the correct path to a sample CSV file.")

Exploring sample file: /Volumes/T7/FCAS_RAISE1SEC-Volume-Bids/PUBLIC_DVD_BIDPEROFFER1_202402010000.CSV

Sample using pandas (first 5 rows, skipping first row):
   I  BIDS  BIDOFFERPERIOD  1     DUID BIDTYPE          TRADINGDATE  \
0  D  BIDS  BIDOFFERPERIOD  1  ADPBA1G  ENERGY  2024/02/01 00:00:00   
1  D  BIDS  BIDOFFERPERIOD  1  ADPBA1G  ENERGY  2024/02/01 00:00:00   
2  D  BIDS  BIDOFFERPERIOD  1  ADPBA1G  ENERGY  2024/02/01 00:00:00   
3  D  BIDS  BIDOFFERPERIOD  1  ADPBA1G  ENERGY  2024/02/01 00:00:00   
4  D  BIDS  BIDOFFERPERIOD  1  ADPBA1G  ENERGY  2024/02/01 00:00:00   

         OFFERDATETIME  PERIODID  MAXAVAIL  ...  BANDAVAIL3  BANDAVAIL4  \
0  2024/01/25 05:06:11         1         6  ...           0           0   
1  2024/01/25 05:06:11         2         6  ...           0           0   
2  2024/01/25 05:06:11         3         6  ...           0           0   
3  2024/01/25 05:06:11         4         6  ...           0           0   
4  2024/01/25 05:06:11         5      

KeyboardInterrupt: 

In [6]:
import os
import pandas as pd
import dask.dataframe as dd
from pathlib import Path
import time

# Set your input and output directories
input_directory = "/Volumes/T7/AEMO_Extracted"
output_directory = str(Path.home() / "Downloads" / "Filtered_RAISE1SEC-2")

# Create output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

print(f"Files will be saved to: {output_directory}")

# Get all CSV files in the directory, filtering out macOS hidden files
csv_files = sorted([f for f in Path(input_directory).glob("*.CSV") 
                    if not f.name.startswith("._")])

if not csv_files:
    raise ValueError(f"No CSV files found in {input_directory}")

print(f"Found {len(csv_files)} CSV files to process")

# Try different encodings to find one that works
encodings_to_try = ['latin1', 'cp1252', 'iso-8859-1', 'utf-8', 'utf-8-sig']
sample_file_path = str(csv_files[0])
print(f"Analyzing sample file structure: {sample_file_path}")

# Try to find a working encoding
working_encoding = None
for encoding in encodings_to_try:
    try:
        print(f"Trying encoding: {encoding}")
        # Add error_bad_lines=False to skip problematic lines
        sample_df = pd.read_csv(
            sample_file_path, 
            encoding=encoding, 
            skiprows=1, 
            nrows=10,
            on_bad_lines='skip'
        )
        if len(sample_df.columns) > 0:  # Make sure we have columns
            working_encoding = encoding
            print(f"Success! Encoding '{encoding}' works.")
            break
        else:
            print(f"Encoding '{encoding}' didn't produce any columns.")
    except UnicodeDecodeError:
        print(f"Encoding '{encoding}' failed with UnicodeDecodeError.")
    except Exception as e:
        print(f"Error with encoding '{encoding}': {str(e)}")

if working_encoding is None:
    raise ValueError("Could not find a suitable encoding for the CSV files.")

print(f"Using encoding: {working_encoding}")
print("Sample columns:", sample_df.columns.tolist())

# Try to identify the BIDTYPE column
bidtype_col = None
for col in sample_df.columns:
    if 'BIDTYPE' in col.upper():
        bidtype_col = col
        print(f"Found BIDTYPE column: {bidtype_col}")
        break

# If we can't find a BIDTYPE column, show all columns
if not bidtype_col:
    print("Could not find a column with 'BIDTYPE' in its name.")
    print("Here are all column names - please identify which one contains the bid type information:")
    for i, col in enumerate(sample_df.columns):
        print(f"  {i}: {col}")
    # Try a common variant
    if 'BID_TYPE' in sample_df.columns:
        bidtype_col = 'BID_TYPE'
        print(f"Using 'BID_TYPE' as the BIDTYPE column")
    else:
        raise ValueError("Please update the code with the correct BIDTYPE column name")

# Function to process a single file
def process_single_file(file_path, bidtype_column, encoding):
    """
    Filter a single CSV file for RAISE1SEC and save as parquet
    
    Parameters:
    file_path (Path): Path to the CSV file
    bidtype_column (str): Name of the column containing bid types
    encoding (str): The encoding to use for reading the CSV
    
    Returns:
    bool: True if successful, False otherwise
    """
    output_filename = f"RAISE1SEC_{file_path.stem}.parquet"
    output_path = Path(output_directory) / output_filename
    
    # Check if file already exists (to support resuming)
    if output_path.exists():
        print(f"\nSkipping {file_path.name} - output already exists at {output_path}")
        return True
    
    print(f"\nProcessing: {file_path.name}")
    print(f"Output will be saved to: {output_path}")
    
    start_time = time.time()
    
    try:
        # First try to load a small sample to verify the file structure
        try:
            sample = pd.read_csv(
                str(file_path), 
                encoding=encoding,
                skiprows=1, 
                nrows=5,
                on_bad_lines='skip'
            )
            # Verify that the BIDTYPE column exists
            if bidtype_column not in sample.columns:
                print(f"Warning: {bidtype_column} column not found in {file_path.name}")
                print(f"Available columns: {sample.columns.tolist()}")
                return False
        except Exception as e:
            print(f"Error sampling {file_path.name}: {str(e)}")
            return False
            
        # Read CSV with Dask, explicitly skipping the first row and using the detected encoding
        ddf = dd.read_csv(
            str(file_path), 
            encoding=encoding,
            skiprows=1, 
            assume_missing=True,
            on_bad_lines='skip'  # Skip problematic lines instead of failing
        )
        
        # Filter for RAISE1SEC (case-insensitive match)
        ddf_filtered = ddf[ddf[bidtype_column].str.upper() == 'RAISE1SEC']
        
        # Save as parquet
        ddf_filtered.to_parquet(str(output_path), write_index=False)
        
        elapsed_time = time.time() - start_time
        print(f"Successfully saved filtered data to {output_path}")
        print(f"Processing time: {elapsed_time:.2f} seconds")
        return True
    except Exception as e:
        print(f"Error processing {file_path.name}: {str(e)}")
        return False

# Process files one at a time automatically
total_files = len(csv_files)
processed_count = 0
failed_files = []

print("\nStarting to process files one by one...")

for file_index, current_file in enumerate(csv_files):
    print(f"\nFile {file_index+1} of {total_files}: {current_file.name}")
    
    success = process_single_file(current_file, bidtype_col, working_encoding)
    
    if success:
        processed_count += 1
    else:
        failed_files.append(current_file.name)
    
    # Optional: add a small delay between files to allow system to recover
    if file_index < total_files - 1:  # Don't delay after the last file
        time.sleep(1)  # 1 second delay

print(f"\nProcessing complete. Successfully processed {processed_count} of {total_files} files.")
if failed_files:
    print(f"Failed to process {len(failed_files)} files:")
    for failed_file in failed_files:
        print(f"  - {failed_file}")
print(f"Filtered files saved to {output_directory}")

Files will be saved to: /Users/danielseymour/Downloads/Filtered_RAISE1SEC-2
Found 3 CSV files to process
Analyzing sample file structure: /Volumes/T7/AEMO_Extracted/PUBLIC_ARCHIVE#BIDPEROFFER_D#FILE01#202408010000.CSV
Trying encoding: latin1
Success! Encoding 'latin1' works.
Using encoding: latin1
Sample columns: ['I', 'BID', 'BIDPEROFFER_D', '3', 'SETTLEMENTDATE', 'DUID', 'BIDTYPE', 'DIRECTION', 'INTERVAL_DATETIME', 'BIDSETTLEMENTDATE', 'OFFERDATE', 'PERIODID', 'VERSIONNO', 'MAXAVAIL', 'FIXEDLOAD', 'ROCUP', 'ROCDOWN', 'ENABLEMENTMIN', 'ENABLEMENTMAX', 'LOWBREAKPOINT', 'HIGHBREAKPOINT', 'BANDAVAIL1', 'BANDAVAIL2', 'BANDAVAIL3', 'BANDAVAIL4', 'BANDAVAIL5', 'BANDAVAIL6', 'BANDAVAIL7', 'BANDAVAIL8', 'BANDAVAIL9', 'BANDAVAIL10', 'LASTCHANGED', 'PASAAVAILABILITY', 'MR_CAPACITY', 'ENERGYLIMIT']
Found BIDTYPE column: BIDTYPE

Starting to process files one by one...

File 1 of 3: PUBLIC_ARCHIVE#BIDPEROFFER_D#FILE01#202408010000.CSV

Processing: PUBLIC_ARCHIVE#BIDPEROFFER_D#FILE01#202408010000.